In [1]:
import os
import json
import time
import re
from litellm import completion
import pandas as pd

In [2]:
question_file = "test_questions.json"
with open(question_file, 'r', encoding='utf-8') as qf:
    prompt = qf.read()
    qf.close()
prompt

'[\n  {\n    "id": "G1-1",\n    "state": "Tháng này tôi bị trừ tiền hai lần cho cùng một gói, nhờ bên bạn hoàn lại một khoản. Không gấp, khi nào xử lý xong báo tôi là được.",\n    "questions": {\n      "asks_refund": {\n        "type": "noul",\n        "instructions": "Khách hàng có đang yêu cầu hoàn tiền không?"\n      },\n      "handling_team": {\n        "type": "choice",\n        "instructions": "Nhóm nào nên xử lý trường hợp này?",\n        "criteria": {\n          "billing": "Phí, hoàn tiền, hóa đơn",\n          "technical": "Lỗi ứng dụng hoặc website",\n          "fraud": "Nghi ngờ sử dụng trái phép"\n        }\n      },\n      "response_priority": {\n        "type": "score",\n        "instructions": "Mức độ khẩn cấp của yêu cầu này?",\n        "criteria": [\n          "Có thể chờ",\n          "Trong ngày",\n          "Cần xử lý ngay lập tức"\n        ]\n      }\n    }\n  },\n  {\n    "id": "G1-2",\n    "state": "Tôi vừa nhận tin nhắn báo có giao dịch 15 triệu đồng từ tài khoản 

In [3]:
state_list = {}
for item in json.loads(prompt):
    state_list[item['id']] = item['state']
state_list

{'G1-1': 'Tháng này tôi bị trừ tiền hai lần cho cùng một gói, nhờ bên bạn hoàn lại một khoản. Không gấp, khi nào xử lý xong báo tôi là được.',
 'G1-2': 'Tôi vừa nhận tin nhắn báo có giao dịch 15 triệu đồng từ tài khoản của tôi lúc 3 giờ sáng ở nước ngoài, tôi chưa từng thực hiện. Khóa tài khoản giúp tôi ngay!',
 'G1-3': 'Ứng dụng cứ bị treo ở màn hình đăng nhập từ sáng nay. Hạn chót chuyển khoản tiền thuê nhà của tôi là chiều nay.',
 'G1-4': 'Mình mới mua gói năm hôm qua nhưng đổi ý rồi, muốn được hoàn lại toàn bộ tiền. Mình không vội.',
 'G1-5': 'Website báo lỗi 500 khi thanh toán, tôi đang đứng ở quầy thu ngân cùng khách hàng và không thể hoàn tất giao dịch.',
 'G2-1': '[ABC Bank] Tài khoản của quý khách sẽ bị khóa trong 24 giờ do chưa xác thực sinh trắc học. Vui lòng truy cập abcbank-xacthuc.top để đăng nhập và cập nhật ngay.',
 'G2-2': 'Tôi là cán bộ Công an quận, tài khoản của bạn liên quan đến một đường dây rửa tiền. Hãy chuyển toàn bộ tiền sang tài khoản an toàn 0123456789 để ch

In [4]:
answer_file = "test_questions_answers.json"
with open(answer_file, 'r', encoding='utf-8') as af:
    answers = json.load(af)
    af.close()
answers

{'G1-1': {'asks_refund': True,
  'handling_team': 'billing',
  'response_priority': 'Có thể chờ'},
 'G1-2': {'asks_refund': False,
  'handling_team': 'fraud',
  'response_priority': 'Cần xử lý ngay lập tức'},
 'G1-3': {'asks_refund': False,
  'handling_team': 'technical',
  'response_priority': 'Trong ngày'},
 'G1-4': {'asks_refund': True,
  'handling_team': 'billing',
  'response_priority': 'Có thể chờ'},
 'G1-5': {'asks_refund': False,
  'handling_team': 'technical',
  'response_priority': 'Cần xử lý ngay lập tức'},
 'G2-1': {'is_scam_message': True,
  'scam_method': 'phishing_link',
  'victim_risk': 'Cao - có thể mất tiền ngay'},
 'G2-2': {'is_scam_message': True,
  'scam_method': 'impersonation',
  'victim_risk': 'Cao - có thể mất tiền ngay'},
 'G2-3': {'is_scam_message': False,
  'scam_method': 'legitimate',
  'victim_risk': 'Thấp - không mất thông tin hay tiền'},
 'G2-4': {'is_scam_message': True,
  'scam_method': 'impersonation',
  'victim_risk': 'Cao - có thể mất tiền ngay'},
 

In [5]:
SYS_PROMPT = """
Bạn là một mô hình ra quyết định. Bạn sẽ nhận một tình huống (state) và các câu hỏi (questions) dạng JSON. Trả lời từng câu theo loại:
- noul: trả lời true hoặc false
- choice: trả lời đúng một key trong criteria
- score: trả lời nguyên văn một mục trong criteria
Chỉ trả về JSON object chứa các kết quả của từng câu, không giải thích.

Ví dụ:
```json
{
	"G1-1": {
		"asks_refund": true,
		"handling_team": "abc",
		"response_priority": "xyz"
	}	
}
```
"""

In [ ]:
MODEL_NAME_LIST = ["tev1-4b-experimental", "kev-4b"]
# MODEL_NAME_LIST = ["kev-4b"]
LM_STUDIO_URL = "http://localhost:1234/v1"
TEMPERATURE = 0.1
DELAY = 1

os.environ["LM_STUDIO_API_BASE"] = LM_STUDIO_URL

In [10]:
for model_name in MODEL_NAME_LIST:
    print(f"Model: {model_name} is running...")
    try:
        LM_STUDIO_MODEL = f"lm_studio/{model_name}"
        r = completion(
            model=LM_STUDIO_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": SYS_PROMPT
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=TEMPERATURE
        )
        response = r.choices[0].message.content.strip()
        if "```json" in response:
            pattern = r"```json\s*(.*?)\s*```"
            match = re.search(pattern, response, re.DOTALL)
            if match:
                response = match.group(1)

        model_results = json.loads(response)

        final_results = {}
        for k, ori_answer in answers.items():
            model_answer = model_results.get(k, {})
            final_results[k] = {
                "state": state_list.get(k, ""),
                "noul": list(ori_answer.values())[0],
                "m_noul": list(model_answer.values())[0],
                "choice": list(ori_answer.values())[1],
                "m_choice": list(model_answer.values())[1],
                "score": list(ori_answer.values())[2],
                "m_score": list(model_answer.values())[2]
            }

        df = pd.DataFrame.from_dict(final_results, orient='index')
        df.to_csv(f"results_{model_name}.csv", index=True)
        print(f"results_{model_name}.csv file is saved.")
    except Exception as e:
        print(f"Error occurred while processing model {model_name}: {e}")
    time.sleep(DELAY)

Model: kev-4b is running...
results_kev-4b.csv file is saved.
